# 05 - Final Model Saving and Inference

This notebook trains the selected best model for each target on all data, saves model files, and exports 30-day forecasts.

In [4]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import LinearRegression
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ARTIFACTS_DIR = BASE_DIR / 'artifacts'
MODELS_DIR = BASE_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

daily = pd.read_csv(ARTIFACTS_DIR / 'daily_series.csv', parse_dates=['date']).set_index('date')
with open(ARTIFACTS_DIR / 'best_models_selected.json', 'r', encoding='utf-8') as f:
    best_models = json.load(f)

classical_tuning = {}
classical_tuning_path = ARTIFACTS_DIR / 'classical_tuning_params.json'
if classical_tuning_path.exists():
    with open(classical_tuning_path, 'r', encoding='utf-8') as f:
        classical_tuning = json.load(f)

ml_tuning = {}
ml_tuning_path = ARTIFACTS_DIR / 'ml_tuning_params.json'
if ml_tuning_path.exists():
    with open(ml_tuning_path, 'r', encoding='utf-8') as f:
        ml_tuning = json.load(f)

HORIZON = 30
SEASONAL_PERIOD = 7

print('Best models:', best_models)
print('XGBoost available:', XGBOOST_AVAILABLE)
print('Classical tuning loaded:', bool(classical_tuning))
print('ML tuning loaded:', bool(ml_tuning))

Best models: {'daily_orders': 'ARIMA', 'daily_revenue': 'ARIMA'}
XGBoost available: True
Classical tuning loaded: True
ML tuning loaded: True


In [5]:
def make_supervised_frame(series, lags=(1,2,3,7,14,28), roll_windows=(7,14)):
    d = pd.DataFrame({'y': series.astype(float)})
    for lag in lags:
        d[f'lag_{lag}'] = d['y'].shift(lag)
    for w in roll_windows:
        d[f'roll_mean_{w}'] = d['y'].shift(1).rolling(w).mean()
        d[f'roll_std_{w}'] = d['y'].shift(1).rolling(w).std()
    idx = d.index
    d['dow'] = idx.dayofweek
    d['is_weekend'] = (idx.dayofweek >= 5).astype(int)
    d['month'] = idx.month
    d['quarter'] = idx.quarter
    return d.dropna()


def make_feature_row(history, dt, lags=(1,2,3,7,14,28), roll_windows=(7,14)):
    row = {}
    for lag in lags:
        row[f'lag_{lag}'] = history.iloc[-lag] if len(history) >= lag else np.nan
    for w in roll_windows:
        vals = history.iloc[-w:] if len(history) >= w else history
        row[f'roll_mean_{w}'] = vals.mean() if len(vals) else np.nan
        row[f'roll_std_{w}'] = vals.std() if len(vals) > 1 else 0.0
    row['dow'] = dt.dayofweek
    row['is_weekend'] = int(dt.dayofweek >= 5)
    row['month'] = dt.month
    row['quarter'] = ((dt.month - 1) // 3) + 1
    return pd.DataFrame([row], index=[dt]).fillna(0.0)


def recursive_forecast(model, train_series, horizon):
    history = train_series.copy()
    idx = pd.date_range(start=history.index[-1] + pd.Timedelta(days=1), periods=horizon, freq='D')
    preds = []
    for dt in idx:
        x = make_feature_row(history, dt)
        p = float(model.predict(x)[0])
        p = max(0.0, p)
        preds.append(p)
        history.loc[dt] = p
    return pd.Series(preds, index=idx)


def save_model(model_obj, path):
    try:
        joblib.dump(model_obj, path)
    except Exception:
        with open(path, 'wb') as f:
            pickle.dump(model_obj, f)


def fit_final(series, model_name, target_name, horizon=30, seasonal_period=7):
    future_idx = pd.date_range(start=series.index[-1] + pd.Timedelta(days=1), periods=horizon, freq='D')
    target_classical = classical_tuning.get(target_name, {})
    target_ml = ml_tuning.get(target_name, {})

    if model_name == 'Naive':
        model = {'type': 'naive'}
        forecast = pd.Series([series.iloc[-1]] * horizon, index=future_idx)
    elif model_name == 'SeasonalNaive':
        base = series.iloc[-seasonal_period:].values
        vals = [base[i % seasonal_period] for i in range(horizon)]
        model = {'type': 'seasonal_naive', 'seasonal_period': seasonal_period}
        forecast = pd.Series(vals, index=future_idx)
    elif model_name == 'Holt':
        model = ExponentialSmoothing(series, trend='add', seasonal=None).fit(optimized=True)
        forecast = model.forecast(horizon)
    elif model_name == 'ETS':
        model = ExponentialSmoothing(series, trend='add', seasonal='add', seasonal_periods=seasonal_period).fit(optimized=True)
        forecast = model.forecast(horizon)
    elif model_name == 'ARIMA':
        order = tuple(target_classical.get('ARIMA', {}).get('order', [1, 1, 1]))
        model = ARIMA(series, order=order).fit()
        forecast = model.forecast(horizon)
    elif model_name == 'SARIMA':
        order = tuple(target_classical.get('SARIMA', {}).get('order', [1, 1, 1]))
        sorder = tuple(target_classical.get('SARIMA', {}).get('seasonal_order', [1, 1, 1, seasonal_period]))
        model = SARIMAX(series, order=order, seasonal_order=sorder, enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        forecast = model.forecast(horizon)
    elif model_name == 'LinearRegression':
        sup = make_supervised_frame(series)
        X, y = sup.drop(columns=['y']), sup['y']
        model = LinearRegression()
        model.fit(X, y)
        forecast = recursive_forecast(model, series, horizon)
    elif model_name == 'XGBoost':
        if not XGBOOST_AVAILABLE:
            raise ImportError('XGBoost is selected but not installed')
        params = target_ml.get('XGBoost') or {
            'n_estimators': 300,
            'learning_rate': 0.05,
            'max_depth': 4,
            'subsample': 0.9,
            'colsample_bytree': 0.9,
            'random_state': 42,
            'objective': 'reg:squarederror'
        }
        sup = make_supervised_frame(series)
        X, y = sup.drop(columns=['y']), sup['y']
        model = XGBRegressor(**params)
        model.fit(X, y)
        forecast = recursive_forecast(model, series, horizon)
    else:
        raise ValueError(model_name)

    forecast = pd.Series(np.maximum(0.0, forecast.values), index=future_idx)
    return model, forecast

In [6]:
rows = []
registry = {}
for target in ['daily_revenue', 'daily_orders']:
    series = daily[target].astype(float)
    model_name = best_models[target]
    model_obj, forecast = fit_final(series, model_name, target_name=target, horizon=HORIZON, seasonal_period=SEASONAL_PERIOD)

    model_path = MODELS_DIR / f'{target}_{model_name}.pkl'
    save_model(model_obj, model_path)

    registry[target] = {
        'model': model_name,
        'classical_tuning': classical_tuning.get(target, {}),
        'ml_tuning': ml_tuning.get(target, {})
    }

    for dt, val in forecast.items():
        rows.append({
            'date': dt,
            'target': target,
            'model': model_name,
            'forecast': float(val)
        })

final_forecasts = pd.DataFrame(rows)
final_forecasts.to_csv(ARTIFACTS_DIR / 'final_forecasts.csv', index=False)

with open(ARTIFACTS_DIR / 'final_model_registry.json', 'w', encoding='utf-8') as f:
    json.dump(registry, f, indent=2)

print('Saved forecasts:', ARTIFACTS_DIR / 'final_forecasts.csv')
print('Saved registry :', ARTIFACTS_DIR / 'final_model_registry.json')
print('Saved models dir:', MODELS_DIR)
final_forecasts.head()

c:\Users\RITIK SINHA\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\RITIK SINHA\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\RITIK SINHA\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\RITIK SINHA\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\RITIK SINHA\AppData\Local\P

Saved forecasts: c:\Users\RITIK SINHA\Downloads\wakmart-timeseries-forecasting\walmart\artifacts\final_forecasts.csv
Saved registry : c:\Users\RITIK SINHA\Downloads\wakmart-timeseries-forecasting\walmart\artifacts\final_model_registry.json
Saved models dir: c:\Users\RITIK SINHA\Downloads\wakmart-timeseries-forecasting\walmart\models


,date,target,model,forecast
0,2025-02-10,daily_revenue,ARIMA,35779.072041
1,2025-02-11,daily_revenue,ARIMA,35727.783146
2,2025-02-12,daily_revenue,ARIMA,35679.640525
3,2025-02-13,daily_revenue,ARIMA,35634.451171
4,2025-02-14,daily_revenue,ARIMA,35592.033918
